In [ ]:
# imports, nødvendige for å kjøre koden 

import os
import json
import logging
import numpy as np
import pandas as pd
import oracledb
from google.cloud import secretmanager
from google.cloud.bigquery import Client

# Trenges når jeg kjører lokal i venv, da bruker jeg mine creds. 
# installer "uv pip install python-dotenv" og opprett en .env fil
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Skriver forstårlig og menneskelig logging --> tid - log level (info, warning etc) - melding

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

In [ ]:
# Henting og håndtering av hemmeligheter. Henter fra GSM og setter dem som miljø variabler

def set_secrets_as_envs():

    logger.info("Loading secrets from Secret Manager")

    client = secretmanager.SecretManagerServiceClient()
    resource_name = f"{os.environ['KNADA_TEAM_SECRET']}/versions/latest"
    response = client.access_secret_version(name=resource_name)
    secret_payload = response.payload.data.decode("UTF-8")
    secrets = json.loads(secret_payload)
    os.environ.update(secrets)
    
    
def get_secrets():

    env = os.getenv("ENV", "prod")

    logger.info(f"Running in environment: {env}")

    set_secrets_as_envs()

    secrets = {
        "user": os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
        "host": os.getenv("DBT_ORCL_HOST"),
        "service": os.getenv("DBT_ORCL_SERVICE"),
        "project_id": os.getenv("BS_PROJECT_ID"),
        "table_uri": os.getenv("BS_TABLE_URI"),
    }

     # Overskriver bruker og passord fra gsm. bruker personlige creds istedet
    if env == "local":

        logger.info("Overriding Oracle credentials with local user")

        secrets["user"] = os.getenv("ORACLE_USER")
        secrets["password"] = os.getenv("ORACLE_PASSWORD")

    return secrets

In [ ]:
# Henter max verdien i kolonnen opprettet fra oracle tabellen, trenges for en delta last!

def get_latest_opprettet(cursor):

    logger.info("Fetching latest (max) opprettet from Oracle")

    sql = """ SELECT MAX(opprettet) FROM brillestonad"""
    cursor.execute(sql)
    max_value = cursor.fetchone()[0]

    logger.info(f"latest opprettet: {max_value}")

    return max_value

In [ ]:
#### E delen av ETL jobben (Extract)

def get_data_from_table(secrets, latest_opprettet=None):

    logger.info("Fetching data from BigQuery")

    client = Client(project=secrets["project_id"])

    # Hvis latest_opprettet is true, altså har en verdig, da henter vi data fra oracle tabellen basert på max(opprettet) -1, 
    # da tar vi med en ekstra dag med data (for å sikre at vi ikke mister noe data)
    if latest_opprettet:

        query = f""" SELECT * FROM `{secrets['table_uri']}`
            WHERE opprettet >= TIMESTAMP_SUB( TIMESTAMP('{latest_opprettet}'), INTERVAL 1 DAY)
        """
    else:

        logger.info("Latest opprettet is empty - full load")

        query = f""" SELECT * FROM `{secrets['table_uri']}`
        """

    logger.info(query)

    df = (client.query(query).to_dataframe().drop_duplicates())

    logger.info(f"Rows fetched: {len(df)}")

    return df

In [ ]:
#### T delen av ETL jobben (Transform)

def transform_data(df):

    logger.info("Starting transformations")

    char_to_replace = {'≥' : '>=', '≤' : '<='}

    if "sats_beskrivelse" in df.columns:
        df["sats_beskrivelse"] = df["sats_beskrivelse"].astype(object)

        for old, new in char_to_replace.items():
            df["sats_beskrivelse"] = df["sats_beskrivelse"].str.replace(old, new, regex=False)

    
    df["opprettet"] = pd.to_datetime(df["opprettet"], errors="coerce").dt.tz_localize(None)
    
    # Force the dataframe into chronological order
    df = df.sort_values(by='opprettet').reset_index(drop=True)
    
    df["periode"] = pd.to_datetime(df["bestillingsdato"], errors="coerce").dt.strftime("%Y%m")

    numeric_cols = ["brillepris", "sats_belop", "belop"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    df = df.replace({np.nan: None})

    logger.info("Transformering er gjonnomført")

    return df

In [ ]:
# Kilden kjører en hel last hver natt. Tabellen kan slettes og re-bygges ved en endring (eks, vårt krav om å legge til en order by på opprettet tid) 
# derfor legger jeg noe kode for validering av datasettet/tabellen

def validate_dataframe(df):

    logger.info("Validering av datasettet")

    required_columns = ["fnr_barn", "belop", "opprettet"]

    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}")

    if df.empty:
        raise ValueError("DataFrame is empty")

    logger.info("Validering compeleted!")

In [ ]:
#### L delen av ETL jobben (Loading)

# lager og returnerer en oracle connection
def create_connection(secrets):

    logger.info("Åpner en connection mot Oracle")

    dsn = oracledb.makedsn(secrets["host"], 1521, service_name=secrets["service"])
    user = f"{secrets['user']}[DVH_FAM_HM]"
    conn = oracledb.connect(user=user, password=secrets["password"], dsn=dsn)

    return conn

def load_dataframe(cursor, df):

    logger.info("Loading dataframe into Oracle")

    insert_sql = """
        INSERT INTO brillestonad (ID, FNR_BARN, FNR_INNSENDER, ORGNR, BESTILLINGSDATO, BRILLEPRIS, BESTILLINGSREFERANSE,
            BEHANDLINGSRESULTAT, SATS, SATS_BELOP, SATS_BESKRIVELSE, BELOP, OPPRETTET, KILDE, PERIODE)

        VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11, :12, :13, :14, :15)
    """

    #rows = list(df.itertuples(index=False, name=None))

    rows = [tuple(
            None if pd.isna(value)
            else value.item() if hasattr(value, "item")
            else value
            for value in row
        )
        for row in df.itertuples(index=False, name=None)
    ]


    cursor.executemany(insert_sql,rows)

    logger.info(f"Inserted rader: {len(rows)}")

In [ ]:
#### Post load

sql_fk_person1 = """
MERGE INTO DVH_FAM_HM.brillestonad T
USING (
    WITH brille AS (
        SELECT
            A.FNR_BARN,
            A.FNR_INNSENDER,
            A.pk_fam_brillestonad,
            A.opprettet,
            TO_CHAR(a.BESTILLINGSDATO,'YYYYMM') periode_ny,

            CASE
                WHEN barn.skjermet_kode = 0
                    THEN barn.fk_person1
                WHEN barn.skjermet_kode != 0
                    THEN -1
                ELSE NULL
            END FK_PERSON1_BARN_NY,

            CASE
                WHEN innsender.skjermet_kode = 0
                    THEN innsender.fk_person1
                WHEN innsender.skjermet_kode != 0
                    THEN -1
                ELSE NULL
            END FK_PERSON1_INNSENDER_NY,

            CASE
                WHEN dim_person_innsender.skjermet_kode = 0
                    THEN dim_person_innsender.pk_dim_person
                WHEN dim_person_innsender.skjermet_kode != 0
                    THEN -1
                ELSE NULL
            END FK_DIM_PERSON_INNSENDER_NY,

            CASE
                WHEN dim_person_barn.skjermet_kode = 0
                    THEN dim_person_barn.pk_dim_person
                WHEN dim_person_barn.skjermet_kode != 0
                    THEN -1
                ELSE NULL
            END FK_DIM_PERSON_BARN_NY

        FROM brillestonad A

        LEFT OUTER JOIN dt_person.ident_off_id_til_fk_person1 barn
            ON barn.off_id = fnr_barn
            AND barn.gyldig_fra_dato < opprettet
            AND barn.gyldig_til_dato >= opprettet

        LEFT OUTER JOIN dt_person.ident_off_id_til_fk_person1 innsender
            ON innsender.off_id = fnr_innsender
            AND innsender.gyldig_fra_dato < opprettet
            AND innsender.gyldig_til_dato >= opprettet

        LEFT OUTER JOIN dt_person.dim_person dim_person_innsender
            ON dim_person_innsender.fk_person1 = innsender.fk_person1
            AND dim_person_innsender.gyldig_fra_dato < opprettet
            AND dim_person_innsender.gyldig_til_dato >= opprettet

        LEFT OUTER JOIN dt_person.dim_person dim_person_barn
            ON dim_person_barn.fk_person1 = barn.fk_person1
            AND dim_person_barn.gyldig_fra_dato < opprettet
            AND dim_person_barn.gyldig_til_dato >= opprettet
    )

    SELECT *
    FROM brille

) S

ON (
    S.PK_FAM_BRILLESTONAD = T.PK_FAM_BRILLESTONAD
)

WHEN MATCHED THEN
UPDATE SET
    T.FK_PERSON1_BARN = S.FK_PERSON1_BARN_NY,
    T.FK_PERSON1_INNSENDER = S.FK_PERSON1_INNSENDER_NY,
    T.FK_DIM_PERSON_INNSENDER = S.FK_DIM_PERSON_INNSENDER_NY,
    T.FK_DIM_PERSON_BARN = S.FK_DIM_PERSON_BARN_NY,
    T.FNR_BARN =
        CASE
            WHEN S.FK_PERSON1_BARN_NY IS NOT NULL
                 OR S.FK_PERSON1_BARN_NY != -1
            THEN NULL
            ELSE T.FNR_BARN
        END,
    T.FNR_INNSENDER =
        CASE
            WHEN S.FK_PERSON1_INNSENDER_NY IS NOT NULL
                 OR S.FK_PERSON1_INNSENDER_NY != -1
            THEN NULL
            ELSE T.FNR_INNSENDER
        END,
    T.OPPDATERT_DATO = LOCALTIMESTAMP
"""


sql_delete_k76 = """
DELETE FROM DVH_FAM_HM.brillestonad
WHERE FK_PERSON1_BARN = -1
"""


sql_barn_info = """
MERGE INTO brillestonad b
USING (
    SELECT
        b2.pk_fam_brillestonad,

        CASE
            WHEN dim_barn.fodt_dato IS NOT NULL
            THEN FLOOR(
                MONTHS_BETWEEN(
                    b2.bestillingsdato,
                    dim_barn.fodt_dato
                ) / 12
            )
        END alder_barn,

        CASE
            WHEN dim_barn.kjonn_nr = 1 THEN 'M'
            WHEN dim_barn.kjonn_nr = 0 THEN 'K'
            ELSE 'U'
        END kjonn_barn,

        dim_barn.fk_dim_geografi_bosted as fk_dim_geografi

    FROM brillestonad b2

    LEFT JOIN dt_person.dim_person dim_barn
        ON dim_barn.fk_person1 = b2.fk_person1_barn
        AND b2.bestillingsdato
            BETWEEN dim_barn.gyldig_fra_dato
            AND dim_barn.gyldig_til_dato

    WHERE b2.bestillingsdato > DATE '0001-01-01'
) src

ON (
    b.pk_fam_brillestonad = src.pk_fam_brillestonad
)

WHEN MATCHED THEN
UPDATE SET
    b.alder_barn = src.alder_barn,
    b.kjonn_barn = src.kjonn_barn,
    b.fk_dim_geografi_barn = src.fk_dim_geografi
"""


sql_update_k76 = """
UPDATE DVH_FAM_HM.brillestonad
SET ORGNR = NULL
WHERE FK_PERSON1_INNSENDER = -1
"""

In [ ]:
def send_data():

    logger.info("Starting pipeline")

    secrets = get_secrets()

    with create_connection(secrets) as conn:
        try:
            with conn.cursor() as cursor:
                latest_opprettet = get_latest_opprettet(cursor) 
                df = get_data_from_table(secrets,latest_opprettet)

                if df.empty:
                    logger.info("No new data found")
                    return

                df = transform_data(df)

                validate_dataframe(df)

                load_dataframe(cursor, df)

                logger.info("Running sql_fk_person1")
                cursor.execute(sql_fk_person1)

                logger.info("Running sql_delete_k76")
                cursor.execute(sql_delete_k76)

                logger.info("Running sql_barn_info")
                cursor.execute(sql_barn_info)

                logger.info("Running sql_update_k76")
                cursor.execute(sql_update_k76)
                conn.commit()

                logger.info("Pipeline completed successfully")

        except Exception as e:

            logger.exception("Pipeline failed")

            conn.rollback()
            raise e

In [ ]:
send_data()